# EDA with SQL

Loads the launch dataset into an in-memory SQLite database and runs the required analysis queries.

In [ ]:
import pandas as pd, sqlite3, json

df = pd.read_csv('spacex_launch_geo.csv')
conn = sqlite3.connect(':memory:')
df.to_sql('SPACEXTBL', conn, index=False, if_exists='replace')
c = conn.cursor()

def q(sql):
    return pd.read_sql(sql, conn)

results = {}

results['sites'] = q("SELECT DISTINCT \"Launch Site\" FROM SPACEXTBL")
results['cca'] = q("SELECT * FROM SPACEXTBL WHERE \"Launch Site\" LIKE 'CCA%' LIMIT 5")
results['nasa_payload'] = q("SELECT SUM(\"Payload Mass (kg)\") as total_payload FROM SPACEXTBL WHERE Customer='NASA (CRS)'")
results['f9v11_avg'] = q("SELECT AVG(\"Payload Mass (kg)\") as avg_payload FROM SPACEXTBL WHERE \"Booster Version\" LIKE 'F9 v1.1%'")
results['first_ground_success'] = q("SELECT MIN(Date) as first_success_ground_pad FROM SPACEXTBL WHERE \"Landing Outcome\"='Success (ground pad)'")
results['drone_4000_6000'] = q("SELECT \"Booster Version\" FROM SPACEXTBL WHERE \"Landing Outcome\" LIKE 'Success%drone ship%' AND \"Payload Mass (kg)\" > 4000 AND \"Payload Mass (kg)\" < 6000")
results['outcomes'] = q("SELECT class, COUNT(*) as total FROM SPACEXTBL GROUP BY class")
results['max_payload_boosters'] = q("SELECT \"Booster Version\" FROM SPACEXTBL WHERE \"Payload Mass (kg)\" = (SELECT MAX(\"Payload Mass (kg)\") FROM SPACEXTBL)")
results['y2015_fail_drone'] = q("SELECT \"Landing Outcome\", \"Booster Version\", \"Launch Site\" FROM SPACEXTBL WHERE \"Landing Outcome\" LIKE 'Failure%drone ship%' AND Date LIKE '2015%'")
results['rank_outcomes'] = q("""SELECT "Landing Outcome", COUNT(*) as cnt FROM SPACEXTBL
    WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY "Landing Outcome" ORDER BY cnt DESC""")

for k, v in results.items():
    print(f"\n=== {k} ===")
    print(v.to_string(index=False))
